# Step 5: Gate (blend Model 1 and Model 2)

Final score for every candidate:  **score = w · P1 + (1 − w) · P2**, where `w` is the weight on Model 1.

1. **P1 and P2:** both models' scores as purchase probabilities (Platt scaling on val2, done in each model's notebook).
2. **Fit the gate on val2:** try `w` from 0 to 1 and keep the best.
   - **Gate A** (as in MOHR, Wang et al. 2024): one `w` for everyone, with and without the short-history rule (`w = 1` for short histories).
   - **Gate C** (our designed gate, features from `gate_features_final.ipynb`): `w` learned from three facts about the user.
   - Optional: **unknown-product rule**, P1 only for products Model 2 never learned.
3. **Evaluate once on test** (`RUN_TEST = True`), with Model 1 alone and Model 2 alone as comparison rows.

`w` is chosen on val2, so val2 numbers are slightly optimistic; test is the honest result.

In [ ]:
# Setup: Colab mounts Drive; locally, Google Drive for desktop. GRP8_PROJECT overrides both.
import os
from pathlib import Path

if "GRP8_PROJECT" in os.environ:
    PROJECT = Path(os.environ["GRP8_PROJECT"])
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT = Path("/content/drive/MyDrive/Grp8Submission")
    except ImportError:
        PROJECT = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/My Drive/Grp8Submission"))[0]
print("PROJECT =", PROJECT)

In [ ]:
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

DATA = PROJECT / "data"
M1_DIR = PROJECT / "models" / "model1"
M2_DIR = PROJECT / "models" / "model2"
GATE_DIR = PROJECT / "features" / "gate"                     # from gate_features_final.ipynb
SEQ_DIR = PROJECT / "features" / "model2" / "sequence inputs"  # Model 2 inputs (optional rule only)
OUT_DIR = PROJECT / "models" / "gate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TEST = False                                  # True only once, when all score files are final
W_GRID = np.round(np.arange(0, 1.0001, 0.05), 2)  # w = weight on Model 1
HISTORY_BUCKETS = [10, 29, 78, 188]               # group edges (past actions); first group = short history
                                                  # (10 = train 10th percentile, same as gate_features_final)

## Metric code
Copied unchanged from `step4_model1_lightgbm.ipynb` (section 2), the same code as the feature, ablation and DIN notebooks, so every model in the report is scored identically.

In [ ]:
METRICS = ["ndcg@10", "hit@10", "hit@1", "mrr"]


def _gains(label, relevance):
    label = np.asarray(label)
    if relevance == "purchase":
        return (label == 3).astype(np.float64)
    if relevance == "graded":
        return (2.0 ** label - 1.0).astype(np.float64)
    raise ValueError("relevance must be 'purchase' or 'graded'")


def _group_layout(sorted_slates):
    # start index of each slate block in an array sorted by slate, and the 1-based rank inside it
    n = len(sorted_slates)
    start = np.r_[0, np.flatnonzero(sorted_slates[1:] != sorted_slates[:-1]) + 1]
    size = np.diff(np.r_[start, n])
    rank = np.arange(n) - np.repeat(start, size) + 1
    return start, rank


def per_slate_metrics(df, score_col, k=10, relevance="purchase", higher_is_better=True, n_tiebreaks=5, seed=0):
    # one row per slate that has a relevant item, with ndcg@k, hit@k, hit@1, mrr
    slate = df["slate_id"].to_numpy()
    gain = _gains(df["label"].to_numpy(), relevance)
    has_rel = pd.Series(gain > 0).groupby(slate).transform("any").to_numpy()
    slate, gain = slate[has_rel], gain[has_rel]
    score = df[score_col].to_numpy(dtype=np.float64)[has_rel]
    if len(slate) == 0:
        raise ValueError("no slates with a relevant item")
    key = score if higher_is_better else -score
    key = np.where(np.isnan(key), -np.inf, key)
    ideal_order = np.lexsort((-gain, slate))
    start, rank = _group_layout(slate[ideal_order])
    disc = np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0)
    idcg = np.add.reduceat(gain[ideal_order] * disc, start)
    rng = np.random.default_rng(seed)
    acc = {m: np.zeros(len(start)) for m in METRICS}
    for _ in range(n_tiebreaks):
        tie = rng.random(len(slate))
        order = np.lexsort((tie, -key, slate))
        g = gain[order]
        rel = g > 0
        acc["ndcg@10"] += np.add.reduceat(g * disc, start) / idcg
        acc["hit@10"] += np.add.reduceat((rel & (rank <= k)).astype(float), start) > 0
        acc["hit@1"] += np.add.reduceat((rel & (rank == 1)).astype(float), start) > 0
        first = np.minimum.reduceat(np.where(rel, rank, np.inf), start)
        acc["mrr"] += 1.0 / first
    out = pd.DataFrame({m: v / n_tiebreaks for m, v in acc.items()})
    out = out.rename(columns={"ndcg@10": f"ndcg@{k}", "hit@10": f"hit@{k}"})
    out.insert(0, "slate_id", slate[ideal_order][start])
    return out


def summarize(per_slate, n_boot=1000, seed=0):
    # mean of each metric with a 95% bootstrap interval over slates
    rng = np.random.default_rng(seed)
    cols = [c for c in per_slate.columns if c != "slate_id"]
    vals = per_slate[cols].to_numpy()
    boot = vals[rng.integers(0, len(vals), size=(n_boot, len(vals)))].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"mean": vals.mean(axis=0), "ci_low": lo, "ci_high": hi}, index=cols)


def paired_diff(a, b, col="ndcg@10", n_boot=2000, seed=0):
    # mean of a - b over the same slates, with a 95% bootstrap interval
    assert (a["slate_id"].to_numpy() == b["slate_id"].to_numpy()).all(), "slates not aligned"
    d = a[col].to_numpy() - b[col].to_numpy()
    rng = np.random.default_rng(seed)
    boot = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"diff": d.mean(), "ci_low": lo, "ci_high": hi}


def seed_avg(tables):
    # average per-slate metrics over seeds (all tables cover the same slates in the same order)
    out = tables[0][["slate_id"]].copy()
    for t in tables[1:]:
        assert (t["slate_id"].to_numpy() == out["slate_id"].to_numpy()).all(), "slates not aligned"
    for m in METRICS:
        out[m] = np.mean([t[m].to_numpy() for t in tables], axis=0)
    return out


def self_test():
    # hand-checkable toy slates (same cases as the feature notebook)
    d = pd.DataFrame({"slate_id": [1, 1, 1, 2, 2, 3, 3], "label": [0, 3, 0, 3, 1, 0, 1],
                      "s": [0.9, 0.5, 0.1, 0.8, 0.2, 0.5, 0.4]})
    ps = per_slate_metrics(d, "s").set_index("slate_id")
    assert list(ps.index) == [1, 2]
    assert np.isclose(ps.loc[1, "ndcg@10"], 1 / np.log2(3)) and ps.loc[1, "mrr"] == 0.5 and ps.loc[1, "hit@1"] == 0
    assert (ps.loc[2] == 1).all()
    d2 = d.assign(s=[0.9, np.nan, 0.1, 0.8, 0.2, 0.5, 0.4])
    assert per_slate_metrics(d2, "s").set_index("slate_id").loc[1, "mrr"] == 1 / 3
    return True


assert self_test()
print("metric self-test OK")

## Step 1: P1 and P2

Load both models' scores, match them product by product (Model 1 uses `wid`, Model 2 uses `position`; the Step 1 file links the two), and convert both to purchase probabilities with each model's own `calibration.json`.

In [ ]:
def load(path, columns=None):
    return pq.read_table(path, columns=columns).to_pandas(ignore_metadata=True)


def platt(x, model_dir):
    c = json.loads((model_dir / "calibration.json").read_text())
    return 1 / (1 + np.exp(-(c["a"] * x + c["b"])))


def load_split(split):
    # Step 1 file: product at each position, and the user's history length
    s1 = load(DATA / f"{split}.parquet", ["slate_id", "candidate_wid_list", "history_type_list"])
    cand = s1[["slate_id", "candidate_wid_list"]].explode("candidate_wid_list").rename(columns={"candidate_wid_list": "wid"})
    cand["wid"] = cand["wid"].astype("int64")
    cand["position"] = cand.groupby("slate_id").cumcount()
    hist_len = pd.Series(s1["history_type_list"].map(len).to_numpy(), index=s1["slate_id"])

    m1 = load(M1_DIR / f"model1_scores_{split}.parquet", ["slate_id", "wid", "label", "raw_score"])
    m2 = load(M2_DIR / f"model2_predictions_{split}.parquet", ["slate_id", "position", "label", "p2_logit"])
    df = (cand.merge(m2, on=["slate_id", "position"], validate="one_to_one")
              .merge(m1, on=["slate_id", "wid"], validate="one_to_one", suffixes=("", "_m1")))
    assert len(df) == len(cand) == len(m1) == len(m2), f"{split}: the two models' files don't match Step 1"
    assert (df["label"] == df["label_m1"]).all(), f"{split}: labels differ between the two models' files"

    df["P1"] = platt(df["raw_score"], M1_DIR)
    df["P2"] = platt(df["p2_logit"], M2_DIR)
    df["history_len"] = df["slate_id"].map(hist_len)
    return df


val2 = load_split("val2")
buy = val2["label"].eq(3)
print(f"val2: {val2['slate_id'].nunique():,} searches, {len(val2):,} candidates, both models matched")
print(f"purchase rate {buy.mean():.5f} | mean P1 {val2['P1'].mean():.5f} | mean P2 {val2['P2'].mean():.5f}")

### How similar are the two models?
Average within-search rank correlation between P1 and P2 (searches with a purchase). Close to 1 means they rank almost the same way, so blending cannot add much.

In [ ]:
v2p = val2[val2.groupby("slate_id")["label"].transform(lambda l: (l == 3).any())]
corr = v2p.groupby("slate_id").apply(lambda g: g["P1"].corr(g["P2"], method="spearman"), include_groups=False)
print(f"rank correlation between Model 1 and Model 2: mean {corr.mean():.3f} over {len(corr):,} searches")

## Step 2: Fit the gate on val2

Every `w` is scored on val2 (`w = 1` is Model 1 alone, `w = 0` is Model 2 alone): one NDCG@10 per search and per `w`. Gate A and Gate A + rule just pick the best `w` from this table.

In [ ]:
def ndcg_table(df, w_values, override=None):
    """Rows = searches with a purchase, columns = w, values = NDCG@10.
    override: optional boolean per candidate; those candidates use P1 only (w = 1)."""
    cols = {}
    for w in w_values:
        score = w * df["P1"] + (1 - w) * df["P2"]
        if override is not None:
            score = score.where(~override, df["P1"])
        cols[w] = per_slate_metrics(df.assign(score=score), "score").set_index("slate_id")["ndcg@10"]
    return pd.DataFrame(cols)


def history_group(hist_len):
    edges = [-np.inf] + HISTORY_BUCKETS + [np.inf]
    names = [f"≤{HISTORY_BUCKETS[0]} (short)"] + [f"{a + 1}–{b}" for a, b in zip(HISTORY_BUCKETS, HISTORY_BUCKETS[1:])] \
            + [f">{HISTORY_BUCKETS[-1]}"]
    return pd.cut(hist_len, edges, labels=names)


T = ndcg_table(v2p, W_GRID)
group = history_group(v2p.groupby("slate_id")["history_len"].first()).loc[T.index]
short = group == group.cat.categories[0]

# Gate A: one w for everyone
curve = T.mean()
w_A = curve.idxmax()

# Gate A + rule: w = 1 for short histories, one w for everyone else
T_rule = pd.DataFrame(np.where(short.to_numpy()[:, None], T[[1.0]].to_numpy(), T.to_numpy()),
                      index=T.index, columns=T.columns)
curve_rule = T_rule.mean()
w_A_rule = curve_rule.idxmax()
print(f"Gate A: w = {w_A:.2f} | Gate A + rule: w = {w_A_rule:.2f} (w = 1 for {short.sum():,} short-history searches)")

plt.figure(figsize=(7, 4))
plt.plot(curve.index, curve.values, label="Gate A: one w for everyone", color="#2a78d6", lw=2)
plt.plot(curve_rule.index, curve_rule.values, label="Gate A + short-history rule", color="#eb6834", lw=2)
plt.scatter([w_A, w_A_rule], [curve[w_A], curve_rule[w_A_rule]], color=["#2a78d6", "#eb6834"], zorder=3)
plt.xlabel("w (weight on Model 1)   ·   w = 0: Model 2 only   ·   w = 1: Model 1 only")
plt.ylabel("val2 NDCG@10")
plt.title("Blend weight vs ranking quality on val2 (dots: best w)", loc="left")
plt.grid(alpha=0.3); plt.legend(frameon=False); plt.tight_layout(); plt.show()

### Gate C: learned gate (`gate_features_final.ipynb`)

The weight depends on three facts about the user (standardised in the gate-features notebook, using train):

**w = sigmoid(a · history length + b · time since last activity + c · purchase count + bias)**, and **w = 1** for short histories.

`a, b, c, bias` are fitted on val2 by minimising the log loss of the blend `w·P1 + (1−w)·P2` against the **purchase** label (label 3), the same label as both models' Platt scaling and the evaluation.

In [ ]:
from scipy.optimize import minimize

GATE_FILES = {"val2": GATE_DIR / "gate_val.parquet", "test": GATE_DIR / "gate_test.parquet"}   # "val" there = val2
Z_COLS = ["gate_z_history_length", "gate_z_time_since_last", "gate_z_purchase_count"]


def gate_inputs(df, split):
    g = load(GATE_FILES[split], ["slate_id", *Z_COLS, "gate_short_history"]).set_index("slate_id")
    assert df["slate_id"].isin(g.index).all(), f"{split}: searches missing from {GATE_FILES[split].name}"
    rows = g.loc[df["slate_id"]]
    X = np.column_stack([rows[Z_COLS].to_numpy(np.float64), np.ones(len(rows))])   # last column = bias
    return X, rows["gate_short_history"].to_numpy() == 1


def gate_c_weight(theta, X, short):
    w = 1 / (1 + np.exp(-X @ theta))
    return np.where(short, 1.0, w)


def blend_log_loss(theta, X, short, P1, P2, y):
    w = gate_c_weight(theta, X, short)
    p = np.clip(w * P1 + (1 - w) * P2, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


X_v2, short_v2 = gate_inputs(val2, "val2")
fit = minimize(blend_log_loss, np.zeros(4), method="L-BFGS-B",
               args=(X_v2, short_v2, val2["P1"].to_numpy(), val2["P2"].to_numpy(), val2["label"].eq(3).to_numpy()))
theta_C = fit.x
print(f"fit converged: {fit.success} | a (history) = {theta_C[0]:+.3f}, b (time since last) = {theta_C[1]:+.3f}, "
      f"c (purchases) = {theta_C[2]:+.3f}, bias = {theta_C[3]:+.3f}")


def gate_c_ndcg(df, split):
    X, s = gate_inputs(df, split)
    w = gate_c_weight(theta_C, X, s)
    ps = per_slate_metrics(df.assign(score=w * df["P1"] + (1 - w) * df["P2"]), "score")
    return ps.set_index("slate_id")["ndcg@10"], pd.Series(w, index=df.index)


C_val2, w_C_val2 = gate_c_ndcg(v2p, "val2")
w_by_search = w_C_val2.groupby(v2p["slate_id"]).first().loc[T.index]
print("\nGate C weight on Model 1, median per history group (val2):")
print(w_by_search.groupby(group, observed=True).median().round(3).to_string())

### Optional: unknown-product rule

Model 2 collapses when the purchased product has no learned item embedding (DIN notebook, Step 9). This rule uses P1 only for candidates Model 2 does not know, and one `w` for the rest. It needs Model 2's sequence inputs (`cand_item` = 1 means unknown); if the file is not found, this row is skipped.

In [ ]:
SEQ_FILES = {s: SEQ_DIR / f"seq_{s}.parquet" for s in ("val2", "test")}


def unknown_to_model2(df, split):
    """True per candidate if Model 2 has no learned embedding for the product (or None if the file is missing)."""
    if not SEQ_FILES[split].exists():
        return None
    seq = load(SEQ_FILES[split], ["slate_id", "cand_item"]).explode("cand_item")
    seq["position"] = seq.groupby("slate_id").cumcount()
    flag = df[["slate_id", "position"]].merge(seq, on=["slate_id", "position"], how="left", validate="one_to_one")
    assert flag["cand_item"].notna().all(), f"{split}: sequence file does not match Step 1"
    return pd.Series(flag["cand_item"].astype(int).to_numpy() == 1, index=df.index)


unk_v2 = unknown_to_model2(v2p, "val2")
if unk_v2 is None:
    print(f"skipped: {SEQ_FILES['val2']} not found")
else:
    T_unk = ndcg_table(v2p, W_GRID, override=unk_v2)
    w_unk = T_unk.mean().idxmax()
    print(f"unknown to Model 2: {unk_v2.mean():.1%} of candidates | best w for the rest = {w_unk:.2f}")

### val2 results

In [ ]:
def gate_rows(T, short, C, T_unk=None):
    """Per-search NDCG@10 for each approach, using the w values fitted on val2."""
    rows = {"Model 1 alone": T[1.0], "Model 2 alone": T[0.0],
            f"Gate A (w = {w_A:.2f})": T[w_A],
            f"Gate A + rule (w = {w_A_rule:.2f})": T[w_A_rule].where(~short, T[1.0]),
            "Gate C (learned w)": C.loc[T.index]}
    if T_unk is not None:
        rows[f"Gate A + unknown-product rule (w = {w_unk:.2f})"] = T_unk[w_unk]
    return rows


def results(rows):
    base, m2 = rows["Model 1 alone"], rows["Model 2 alone"]
    table = lambda v: v.rename("ndcg@10").rename_axis("slate_id").reset_index()
    out = []
    for name, v in rows.items():
        r = paired_diff(table(v), table(base))
        out.append({"": name, "NDCG@10": v.mean(), "vs Model 1": r["diff"],
                    "95% CI": f"{r['ci_low']:+.4f} to {r['ci_high']:+.4f}",
                    "vs Model 2": v.mean() - m2.mean()})
    return pd.DataFrame(out).set_index("")


val2_rows = gate_rows(T, short, C_val2, T_unk if unk_v2 is not None else None)
val2_table = results(val2_rows)
print(val2_table.round(4).to_string())
print("\nNDCG@10 by history group (val2):")
print(pd.DataFrame(val2_rows).groupby(group, observed=True).mean().round(4).to_string())

FINAL_GATE = "Gate C (learned w)"   # decide after reading the val2 table: the simplest gate not clearly worse than the best

## Step 3: Evaluate once on test

Uses everything fitted on val2 (calibration, `w`, Gate C's `a, b, c, bias`); nothing is fitted on test. Set `RUN_TEST = True` (top of the notebook) only when every score file is final, and run this once.

In [ ]:
if not RUN_TEST:
    print("RUN_TEST is False: test not scored yet.")
else:
    test = load_split("test")
    tp = test[test.groupby("slate_id")["label"].transform(lambda l: (l == 3).any())]
    T_test = ndcg_table(tp, sorted({1.0, 0.0, w_A, w_A_rule}))
    group_t = history_group(tp.groupby("slate_id")["history_len"].first()).loc[T_test.index]
    short_t = group_t == group_t.cat.categories[0]
    C_test, _ = gate_c_ndcg(tp, "test")
    unk_t = unknown_to_model2(tp, "test") if unk_v2 is not None else None
    T_unk_test = ndcg_table(tp, [w_unk], override=unk_t) if unk_t is not None else None

    test_rows = gate_rows(T_test, short_t, C_test, T_unk_test)
    test_table = results(test_rows)
    print(f"TEST, {len(T_test):,} searches with a purchase (fitted on val2; final gate: {FINAL_GATE})")
    print(test_table.round(4).to_string())
    print("\nNDCG@10 by history group (test):")
    print(pd.DataFrame(test_rows).groupby(group_t, observed=True).mean().round(4).to_string())
    test_table.to_csv(OUT_DIR / "gate_test_results.csv")

## Save

In [ ]:
val2_table.to_csv(OUT_DIR / "gate_val2_results.csv")
curve.rename("ndcg@10").to_csv(OUT_DIR / "gate_weight_curve_val2.csv")
config = {"final_gate": FINAL_GATE, "fitted_on": "val2",
          "gate_A_w": float(w_A), "gate_A_rule_w": float(w_A_rule), "short_history_max_actions": HISTORY_BUCKETS[0],
          "gate_C": {"a_history": theta_C[0], "b_time_since_last": theta_C[1], "c_purchases": theta_C[2],
                     "bias": theta_C[3], "inputs": Z_COLS, "label": "purchase (label 3)"},
          "unknown_product_rule_w": None if unk_v2 is None else float(w_unk),
          "model_rank_correlation_val2": float(corr.mean())}
(OUT_DIR / "gate_config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=float))
print("saved to", OUT_DIR)